# The same rule under different execution models

Fill assumptions change results. Run one rule under a frictionless model, a queue model, latency with impact and the strict preset, then compare costs and returns.

In [ ]:
# A temporary workspace and a synthetic daily recording, so the notebook runs anywhere.
import csv, datetime, json, random
from pathlib import Path
from tempfile import mkdtemp

from sim_platform import Platform

folder = Path(mkdtemp(prefix='market-lab-notebook-'))
rng = random.Random(7)
price = 100.0
recording = folder / 'acme.csv'
with recording.open('w', newline='') as stream:
    out = csv.writer(stream)
    out.writerow(['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
    for day in range(40):
        date = datetime.date(2021, 3, 1) + datetime.timedelta(days=day)
        close = price * (1 + rng.gauss(0.0005, 0.012))
        out.writerow([date.isoformat(), round(price, 2), round(max(price, close) + 0.5, 2),
                      round(min(price, close) - 0.5, 2), round(close, 2), 1_000_000])
        price = close
app = Platform(folder / 'workspace')
profile = app.create_profile('Notebook')
print(recording)

In [ ]:
from sim_platform.execution import ExecutionConfig
from sim_platform.reference_agents import execution_preset

DAY = 86_400 * 10**9
models = {
    'frictionless': ExecutionConfig(),
    'queue': ExecutionConfig(model='displayed-limit-queue-v1'),
    'latency+impact': ExecutionConfig(latency_ns=50_000_000, impact_bps=25, fee_per_share=50),
    'strict preset': execution_preset('strict'),
}
results = {}
for name, config in models.items():
    account = app.create_account(profile, name, 1_000_000_000)
    experiment = app.open_experiment(profile, account, source=recording, method='rule',
                                     execution_mode='controlled', execution_config=config,
                                     settings=dict(quantity=500, hold_ns=5 * DAY))
    while experiment.runner.active:
        experiment.step()
    results[name] = experiment.run_id
    experiment.close()
for name, run_id in results.items():
    figures = app.pnl_attribution(profile, run_id)
    print(f"{name:15} net {figures['pnl']['net'] / 1e4:>10.2f} USD  "
          f"costs {figures['costs']['total'] / 1e4:>8.2f} USD")

In [ ]:
diff = app.diff_runs(profile, results['frictionless'], results['strict preset'])
for change in diff['execution']:
    print(change['field'], change['a'], '->', change['b'])